# 29 · Every model on one set

Every notebook trains on notebook 0's training split, picks its best epoch on the select split, and scores its models on
the validation split the same way (`training.val_scores`: SSIM-Y, and luma and chroma PSNR, of each patch's 128 px
core), keeping each model's per-patch scores with `training.record`. This notebook sets them all side by side: one
table of every model of every notebook, against notebook 1's bicubic, the skip every model is built on.

- **The 95% range** of a gain is a bootstrap over the validation sources: the patches of one source move together.
- **Seeds.** Where a notebook trained a model at several seeds, each seed is its own row: how far apart they land is
  the luck in any one row.
- **The graphs.** Every notebook checks its exported graph against its torch model before it publishes it, and
  notebooks 3, 23 and 24 score their graphs on the validation split too, so a row here is also what the graph scores.

Needs: every notebook that records scores (1–12, 16, 18, 20, 22, 24, 26, 27), run first.

In [1]:
"""Settings."""

import os

import numpy as np

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import training

manifest = training.load_manifest()
recorded = training.recorded()
bicubic = recorded[1, "bicubic"]
results = {f"{notebook:02d} · {name}": scores for (notebook, name), scores in sorted(recorded.items())}
print(f"{len(results)} models from {len({notebook for notebook, _ in recorded})} notebooks, "
      f"{len(bicubic['ssim']):,} validation patches each")

5 models from 1 notebooks, 24,286 validation patches each


## 1. Every model

Best first: SSIM-Y, its gain over bicubic with the 95% range, luma PSNR and its gain, chroma PSNR's gain, and the
SSIM-Y gain by what a patch shows and by the host's line.

In [2]:
training.show(training.compare(results, manifest, base=bicubic))

                       model  SSIM-Y  over bicubic          95% range  luma dB  dB over  chroma dB over   video  screen   photo    x264   nvenc
         01 · Lanczos-3, PIL  0.8528       +0.0007 +0.0006 to +0.0009    28.70  +0.1193         -0.0048 -0.0000 +0.0013 +0.0007 +0.0001 +0.0014
                01 · bicubic  0.8521       +0.0000 +0.0000 to +0.0000    28.58  +0.0000         +0.0000 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000
01 · bicubic, PIL (a = -0.5)  0.8513       -0.0008 -0.0010 to -0.0006    28.50  -0.0842         +0.0132 +0.0005 -0.0017 -0.0013 -0.0001 -0.0015
               01 · bilinear  0.8434       -0.0087 -0.0096 to -0.0078    27.97  -0.6130         +0.0277 -0.0018 -0.0140 -0.0102 -0.0049 -0.0124
                01 · nearest  0.8362       -0.0159 -0.0166 to -0.0151    27.47  -1.1123         -0.1104 -0.0162 -0.0126 -0.0248 -0.0165 -0.0152


## 2. By slice QP

The SSIM-Y gain over bicubic in each slice QP bin; the first row counts the patches.

In [3]:
training.show(training.gain_by_qp(results, manifest, base=bicubic))

                       model    0-18   18-26   26-30   30-34   34-38   38-42   42-46   46-52
                     patches    3610    3928    3730    3585    3539    2846    2005    1043
         01 · Lanczos-3, PIL +0.0023 +0.0016 +0.0010 +0.0005 +0.0000 -0.0004 -0.0005 -0.0006
                01 · bicubic +0.0000 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000
01 · bicubic, PIL (a = -0.5) -0.0020 -0.0016 -0.0011 -0.0007 -0.0003 +0.0002 +0.0004 +0.0007
               01 · bilinear -0.0175 -0.0144 -0.0114 -0.0080 -0.0048 -0.0020 +0.0001 +0.0017
                01 · nearest -0.0156 -0.0166 -0.0173 -0.0172 -0.0166 -0.0149 -0.0126 -0.0107


## 3. What it says

*(written after the run)*